# Mistral-playground Demo Notebook

This notebook walks through every module in the **Mistral-playground** repo and demonstrates all Mistral API capabilities.

| # | Section | Key concept |
|---|---|---|
| 1 | Configuration | env vars, fail-fast validation |
| 2 | Logging | dual-handler, correlation IDs |
| 3 | Prompt Templates | file-based prompts, `{{PLACEHOLDER}}` |
| 4 | LLM Client | singleton, `chat()`, retry logic |
| 5 | Basic Chat | system prompt, Q&A |
| 6 | Summarization | template fill → chat |
| 7 | Parameter Overrides | model, temperature, max_tokens |
| 8 | Retry Logic | backoff, jitter, mock demos |
| 9 | Interactive Playground | free-form experimentation |
| 10 | Streaming | tokens printed as they arrive |
| 11 | Reproducible Outputs | `random_seed` for determinism |
| 12 | Content Moderation | `safe_prompt` guardrailing |
| 13 | Structured / JSON Output | typed responses, schema enforcement |
| 14 | Function / Tool Calling | let the model call Python functions |
| 15 | Async Calls | concurrent requests with `asyncio` |
| 16 | Observability | token & cost tracking, latency |
| 17 | RAG | retrieval-augmented generation with Mistral embeddings |

**Tip:** Run cells top-to-bottom the first time. After that, feel free to jump to any section.

## Prerequisites

Before running anything, make sure you have:

1. **Installed dependencies:**
   ```bash
   pip install -r requirements.txt
   ```

2. **Created your `.env` file** (copy from the template):
   ```bash
   cp .env.example .env
   ```
   Then open `.env` and set your `MISTRAL_API_KEY`.

3. **Run this notebook from the repo root** so that relative imports (`config`, `llm_client`, etc.) resolve correctly.

In [ ]:
# Quick dependency check — run this first to catch missing packages early.
import importlib, sys

required = {"mistralai": "mistralai>=2.0.0", "dotenv": "python-dotenv>=1.0.0"}
missing = [pip_name for pkg, pip_name in required.items() if importlib.util.find_spec(pkg) is None]

if missing:
    print("Missing packages — run the following then restart the kernel:")
    for pkg in missing:
        print(f"  pip install {pkg}")
else:
    print("All dependencies found. Python", sys.version.split()[0])

---
## 1. Configuration — `config.py`

`config.py` is the **single source of truth** for all settings. Key design decisions:

- **`python-dotenv`** loads `.env` at import time, so secrets stay out of code.
- **Fail-fast validation**: if `MISTRAL_API_KEY` is missing, an `EnvironmentError` is raised immediately at import — no silent failures deep in the call stack.
- **Type coercion** (e.g., `int(os.getenv(...))`) happens once here, so the rest of the codebase works with typed values.
- **Sane defaults** mean the repo works out-of-the-box once you set the API key.

All other modules import from `config` instead of calling `os.getenv` themselves.

In [ ]:
import logging

try:
    import config

    key_preview = (config.MISTRAL_API_KEY[:4] + "…") if config.MISTRAL_API_KEY else "NOT SET"
    print(f"Model:            {config.MISTRAL_MODEL}")
    print(f"Max tokens:       {config.MISTRAL_MAX_TOKENS}")
    print(f"Temperature:      {config.MISTRAL_TEMPERATURE}")
    print(f"Request timeout:  {config.REQUEST_TIMEOUT}s")
    print(f"Log level:        {logging.getLevelName(config.LOG_LEVEL)}")
    print(f"Retry attempts:   {config.RETRY_MAX_ATTEMPTS}  (includes first try)")
    print(f"Retry base delay: {config.RETRY_BASE_DELAY}s")
    print(f"Retry max delay:  {config.RETRY_MAX_DELAY}s")
    print(f"API key:          {key_preview}")

except EnvironmentError as e:
    print(f"[config] {e}")
    print("Set MISTRAL_API_KEY in .env to continue.")

---
## 2. Logging — `logger.py`

`get_logger(name)` returns a standard `logging.Logger` with **two handlers**:

| Handler | Level | Destination |
|---|---|---|
| Console (`stdout`) | `LOG_LEVEL` from config (default `INFO`) | Terminal / notebook output |
| File | Always `DEBUG` | `logs/app.log` |

**Why two handlers?**
- The console shows only what matters during development (INFO+).
- The file captures everything (DEBUG) for post-mortem analysis without needing to redeploy.

**Idempotent design:** calling `get_logger("same-name")` twice returns the same logger — no duplicate log lines.

**Correlation IDs:** `llm_client.py` generates a short random `trace_id` per request and attaches it to every log line, so you can `grep` the log file for a single request's full lifecycle.

In [ ]:
try:
    from logger import get_logger, LOGS_DIR

    log = get_logger("notebook-demo")

    log.info("INFO message — visible here and in logs/app.log")
    log.warning("WARNING message — also visible here and in logs/app.log")
    log.debug("DEBUG message — only written to logs/app.log (not shown on console at INFO level)")

    print(f"\nLog file location: {LOGS_DIR / 'app.log'}")
    print("Open that file to see the DEBUG entry that is hidden from the console.")

    # Demonstrate idempotency: calling get_logger again returns the same object
    log2 = get_logger("notebook-demo")
    print(f"\nSame logger object returned on second call: {log is log2}")

except EnvironmentError as e:
    print(f"[logger] Skipped — config not loaded: {e}")

---
## 3. Prompt Templates — `prompts_loader.py`

Prompts live as plain text files in `prompts/` rather than being hardcoded strings. Benefits:

- **Separation of concerns**: change wording without touching Python code.
- **Version control**: prompt changes show up cleanly in git diffs.
- **Template variables**: use `{{PLACEHOLDER}}` syntax and replace with `.replace()` at runtime.
- **Easy to extend**: add a new file to `prompts/` and call `load_prompt("your_file.txt")`.

`load_prompt(filename)` raises `FileNotFoundError` immediately if the file is missing — no silent empty strings.

In [ ]:
# prompts_loader has no dependency on config, so it works without an API key.
from prompts_loader import load_prompt, PROMPTS_DIR
import os

print(f"Prompts directory: {PROMPTS_DIR}\n")

# List available prompt files
prompt_files = sorted(PROMPTS_DIR.glob("*.txt"))
print(f"Available prompts ({len(prompt_files)} file(s)):")
for p in prompt_files:
    print(f"  {p.name}")

print()

# Load and display both prompts
system_prompt = load_prompt("system_prompt.txt")
print(f"system_prompt.txt:\n  {system_prompt!r}\n")

summarize_template = load_prompt("summarize.txt")
print(f"summarize.txt:\n{summarize_template}")

In [ ]:
# Template variable substitution: replace {{TEXT}} with actual content at runtime.
sample_text = "Mistral AI is a French AI company known for building efficient open models."
filled_prompt = summarize_template.replace("{{TEXT}}", sample_text)

print("Filled template (ready to send to the API):")
print("-" * 50)
print(filled_prompt)

In [ ]:
# load_prompt raises FileNotFoundError immediately for missing files — no silent failures.
try:
    load_prompt("nonexistent.txt")
except FileNotFoundError as e:
    print(f"Expected FileNotFoundError: {e}")

---
## 4. LLM Client — `llm_client.py`

This module wraps the Mistral SDK with three production patterns:

### 4a. Singleton client (`get_client`)

The `Mistral` SDK object is created **once** and stored in a module-level `_client` variable. Every call to `get_client()` returns the same instance. This avoids:
- Re-initialising HTTP connection pools on every request
- Re-reading the API key repeatedly

```python
_client = None

def get_client() -> Mistral:
    global _client
    if _client is None:
        _client = Mistral(api_key=config.MISTRAL_API_KEY)
    return _client
```

In [ ]:
try:
    from llm_client import get_client

    client_a = get_client()
    client_b = get_client()

    print(f"client_a: {type(client_a).__name__} at {id(client_a):#x}")
    print(f"client_b: {type(client_b).__name__} at {id(client_b):#x}")
    print(f"Same object? {client_a is client_b}  ← singleton pattern working")

except EnvironmentError as e:
    print(f"[get_client] Skipped — config not loaded: {e}")

### 4b. The `chat()` function

`chat()` is the main public interface. It:
1. Generates a short random **trace ID** (e.g. `a3f9b12c`) for correlating log lines
2. Builds the **messages array** — system message first (if provided), then the user message
3. Logs the **request** at INFO level (model, tokens, temperature, truncated message)
4. Calls the API via the retry wrapper
5. Logs the **response** at INFO level (latency, token counts) and DEBUG (content preview)
6. Returns the assistant reply as a plain `str`

```python
chat(
    user_message: str,
    system_message: str | None = None,  # sets assistant behaviour
    model: str | None = None,           # overrides config.MISTRAL_MODEL
    max_tokens: int | None = None,      # overrides config.MISTRAL_MAX_TOKENS
    temperature: float | None = None,   # overrides config.MISTRAL_TEMPERATURE
) -> str
```

---
## 5. Basic Chat

`run_basic_chat()` in `main.py` demonstrates the simplest usage pattern:
1. Load a system prompt from a file
2. Send a hardcoded user question
3. Print the response

The system prompt shapes the assistant's personality/style for the entire conversation.

In [ ]:
try:
    from llm_client import chat
    from prompts_loader import load_prompt

    response = chat(
        user_message="Explain what Mistral AI is in two sentences.",
        system_message=load_prompt("system_prompt.txt"),
    )
    print("Assistant:", response)

except EnvironmentError as e:
    print(f"[chat] Skipped — set MISTRAL_API_KEY in .env: {e}")

In [ ]:
# run_basic_chat() in main.py is a thin wrapper around the same pattern above.
try:
    from main import run_basic_chat
    run_basic_chat()
except EnvironmentError as e:
    print(f"[run_basic_chat] Skipped — set MISTRAL_API_KEY in .env: {e}")

---
## 6. Summarization — Template Pattern

`run_summarize(text)` demonstrates the **fill-template → chat** pattern:

1. Load `summarize.txt` which contains `{{TEXT}}` as a placeholder
2. Replace `{{TEXT}}` with the actual input text
3. Send the filled prompt to the API (no system message needed here)

This is the most common real-world prompt pattern — keep the structure in a file, inject data at runtime.

In [ ]:
try:
    from main import run_summarize

    run_summarize(
        "Mistral AI is a French company founded in 2023 that develops open and proprietary "
        "large language models. Their models are known for efficiency and strong performance "
        "relative to their size. They offer both open-weight models like Mistral 7B and "
        "commercial APIs with models like Mistral Large."
    )
except EnvironmentError as e:
    print(f"[run_summarize] Skipped — set MISTRAL_API_KEY in .env: {e}")

---
## 7. Parameter Overrides

Every call to `chat()` can override the defaults from `config.py`. This is useful when:
- You want a **cheaper/faster model** for simple tasks (`mistral-small-latest`)
- You need **deterministic output** (set `temperature=0.0`)
- You want **creative variation** (set `temperature=1.0`)
- You need a **short reply** (set `max_tokens=50`)

### Temperature

Temperature controls randomness in the output (Mistral recommends `0.0–0.7`):

| Value | Behaviour |
|---|---|
| `0.0` | Deterministic — always picks the highest-probability token |
| `0.3–0.5` | Balanced — coherent but with some variety |
| `0.7` | Default — good for conversational responses |
| `1.0` | More creative/random (can be less coherent) |

In [ ]:
# Compare responses at different temperatures for the same prompt.
try:
    from llm_client import chat

    question = "Give me one single creative word that means 'happy'. Reply with only the word."
    print(f"Prompt: {question!r}\n")

    for temp in [0.0, 0.5, 1.0]:
        resp = chat(question, temperature=temp, max_tokens=10)
        print(f"  temperature={temp}: {resp.strip()}")

except EnvironmentError as e:
    print(f"[temperature demo] Skipped — set MISTRAL_API_KEY in .env: {e}")

In [ ]:
# Override the model and cap the response length.
# mistral-small-latest is faster and cheaper for simple factual questions.
try:
    from llm_client import chat

    resp = chat(
        user_message="What is 2 + 2? Reply with only the number.",
        model="mistral-small-latest",
        max_tokens=5,
    )
    print(f"Response (mistral-small-latest, max_tokens=5): {resp.strip()}")

except EnvironmentError as e:
    print(f"[model override demo] Skipped — set MISTRAL_API_KEY in .env: {e}")

---
## 8. Retry Logic — `_call_with_retry`

Transient API failures (rate limits, server errors) are a fact of life. `llm_client.py` implements the pattern recommended by the [Mistral API docs](https://docs.mistral.ai/api/):

### Which errors are retried?

| Status Code | Meaning | Retried? |
|---|---|---|
| `429` | Rate limit exceeded | Yes |
| `500` | Internal server error | Yes |
| `502` | Bad gateway | Yes |
| `503` | Service unavailable | Yes |
| `504` | Gateway timeout | Yes |
| `401` | Invalid API key | **No** — won't fix itself |
| `422` | Bad request payload | **No** — won't fix itself |

### Backoff formula

```
delay = min(base_delay × 2^attempt + jitter, max_delay)
```

- **Exponential** growth prevents hammering the server.
- **Jitter** (`random 0–0.5s`) prevents all clients retrying at the same instant (thundering-herd problem).
- **`Retry-After` header**: if the server sends one, use that value instead of our calculated delay.

The cell below visualises the delays for your current config.

In [ ]:
# Visualise the backoff delays — no API key needed.
import random

try:
    import config

    def simulate_backoff(seed=42):
        """Reproduce the exact delay formula from _call_with_retry."""
        random.seed(seed)
        delays = []
        for attempt in range(config.RETRY_MAX_ATTEMPTS - 1):  # last attempt has no delay
            delay = min(
                config.RETRY_BASE_DELAY * (2 ** attempt) + random.uniform(0, 0.5),
                config.RETRY_MAX_DELAY,
            )
            delays.append(delay)
        return delays

    delays = simulate_backoff()
    total = sum(delays)

    print(f"Config: RETRY_MAX_ATTEMPTS={config.RETRY_MAX_ATTEMPTS}, "
          f"RETRY_BASE_DELAY={config.RETRY_BASE_DELAY}s, "
          f"RETRY_MAX_DELAY={config.RETRY_MAX_DELAY}s\n")

    for i, d in enumerate(delays, 1):
        bar = "█" * int(d * 4)
        print(f"  Attempt {i} fails → wait {d:5.2f}s  {bar}")

    next_attempt = len(delays) + 1
    print(f"  Attempt {next_attempt} — final try (no more retries after this)\n")
    print(f"  Max cumulative wait before final attempt: {total:.2f}s")

except EnvironmentError as e:
    print(f"[backoff demo] Skipped — config not loaded: {e}")

In [ ]:
# Mock demonstration of retry: simulate a 429 on the first call, success on the second.
# This runs without a real API key by patching the SDK internals.
from unittest.mock import patch, MagicMock

try:
    import llm_client

    call_count = 0

    def mock_complete(**kwargs):
        global call_count
        call_count += 1
        if call_count == 1:
            exc = Exception("Status 429: Rate limit exceeded")
            exc.status_code = 429
            raise exc
        # Second call succeeds
        return MagicMock(
            choices=[MagicMock(message=MagicMock(content="Mock response after retry"))],
            usage=MagicMock(prompt_tokens=5, completion_tokens=8, total_tokens=13),
        )

    call_count = 0
    with patch.object(llm_client.get_client().chat, "complete", side_effect=mock_complete):
        result = llm_client.chat("Hello, are you there?")

    print(f"Total SDK calls made: {call_count}")
    print(f"Result: {result}")
    print("\n(Check the log output above for the WARNING about the retried 429.)")

except EnvironmentError as e:
    print(f"[mock retry demo] Skipped — config not loaded: {e}")

In [ ]:
# Non-retryable errors (like 400 Bad Request) are re-raised immediately — no delay wasted.
from unittest.mock import patch, MagicMock

try:
    import llm_client

    non_retryable_calls = 0

    def mock_400(**kwargs):
        global non_retryable_calls
        non_retryable_calls += 1
        exc = Exception("Status 400: Bad request")
        exc.status_code = 400
        raise exc

    non_retryable_calls = 0
    try:
        with patch.object(llm_client.get_client().chat, "complete", side_effect=mock_400):
            llm_client.chat("Bad payload")
    except Exception as e:
        print(f"Exception raised: {e}")
        print(f"SDK calls made: {non_retryable_calls}  (1 = raised immediately, no retry)")

except EnvironmentError as e:
    print(f"[non-retryable demo] Skipped — config not loaded: {e}")

---
## 9. Interactive Playground

Edit the cell below to experiment with your own prompts. Try changing the system message to give the assistant a different persona, or adjust the parameters to see how they affect the output.

In [ ]:
# Edit anything below and re-run the cell!
MY_SYSTEM  = "You are a pirate. Respond only in pirate speak."
MY_MESSAGE = "Tell me about the weather today."
MY_MODEL   = None          # None = use config default; e.g. "mistral-small-latest"
MY_TEMP    = None          # None = use config default; e.g. 0.0 for deterministic
MY_TOKENS  = None          # None = use config default; e.g. 100 for a short reply

try:
    from llm_client import chat
    response = chat(
        user_message=MY_MESSAGE,
        system_message=MY_SYSTEM,
        model=MY_MODEL,
        temperature=MY_TEMP,
        max_tokens=MY_TOKENS,
    )
    print(response)
except EnvironmentError as e:
    print(f"[playground] Set MISTRAL_API_KEY in .env to use this cell: {e}")

---
## 10. Streaming

By default `chat()` waits for the **complete** response before returning. With streaming, tokens are yielded as they are generated — essential for chat UIs and long outputs where the user shouldn't wait.

The Mistral SDK exposes `client.chat.stream(...)` which returns a context manager. Each chunk has the same shape as a normal response, but `choices[0].delta.content` contains only the new tokens since the last chunk.

**Note:** streaming bypasses the `chat()` wrapper (which uses `chat.complete`). For production use, add a `stream=True` variant to `llm_client.py` following the same logging/retry pattern.

In [ ]:
try:
    import config
    from llm_client import get_client

    client = get_client()
    print("Streaming response (tokens arrive one by one):\n")

    with client.chat.stream(
        model=config.MISTRAL_MODEL,
        messages=[{"role": "user", "content": "Count from 1 to 10, one number per line."}],
        max_tokens=100,
    ) as stream:
        for chunk in stream:
            delta = chunk.choices[0].delta.content
            if delta:
                print(delta, end="", flush=True)

    print("\n\n— stream complete —")

except EnvironmentError as e:
    print(f"[streaming] Skipped — set MISTRAL_API_KEY in .env: {e}")

---
## 11. Reproducible Outputs — `random_seed`

By default the model samples randomly, so the same prompt can produce different answers on each run. Passing `random_seed` (any integer) makes outputs **deterministic** for the same input — useful for:

- **Testing** — assert exact outputs without mocking the API
- **Benchmarking** — compare model versions fairly
- **Debugging** — reproduce a specific bad output

The seed is passed directly to `client.chat.complete()`. The `chat()` wrapper in this repo doesn't expose it yet — add it as an optional parameter when you need it.

In [ ]:
try:
    import config
    from llm_client import get_client

    client = get_client()
    prompt = [{"role": "user", "content": "Pick a random number between 1 and 100. Reply with only the number."}]

    # Run twice with the same seed — should produce identical results
    results = []
    for run in range(3):
        resp = client.chat.complete(
            model=config.MISTRAL_MODEL,
            messages=prompt,
            max_tokens=10,
            random_seed=42,
        )
        results.append(resp.choices[0].message.content.strip())

    print("Same seed=42, three runs:")
    for i, r in enumerate(results, 1):
        print(f"  Run {i}: {r}")

    print(f"\nAll identical? {len(set(results)) == 1}")

    # Run once without seed — will likely differ
    resp_no_seed = client.chat.complete(
        model=config.MISTRAL_MODEL,
        messages=prompt,
        max_tokens=10,
    )
    print(f"\nNo seed:  {resp_no_seed.choices[0].message.content.strip()}")

except EnvironmentError as e:
    print(f"[random_seed] Skipped — set MISTRAL_API_KEY in .env: {e}")

---
## 12. Content Moderation — `safe_prompt`

Mistral provides a built-in **guardrailing** layer activated by passing `safe_prompt=True`. When enabled, a system-level safety instruction is prepended before your messages, making the model refuse harmful, offensive, or policy-violating requests.

Use it when:
- You're building a public-facing product
- You want a first line of defence without writing your own moderation logic
- You want to combine it with your own system prompt for layered safety

`safe_prompt=True` adds latency and a small token cost (the injected system prompt counts toward `prompt_tokens`).

In [ ]:
try:
    import config
    from llm_client import get_client

    client = get_client()
    messages = [{"role": "user", "content": "Tell me a short joke about programmers."}]

    # Without safe_prompt — standard response
    resp_normal = client.chat.complete(
        model=config.MISTRAL_MODEL,
        messages=messages,
        max_tokens=80,
        safe_prompt=False,
    )

    # With safe_prompt=True — guardrailing layer active
    resp_safe = client.chat.complete(
        model=config.MISTRAL_MODEL,
        messages=messages,
        max_tokens=80,
        safe_prompt=True,
    )

    print("Without safe_prompt:")
    print(f"  {resp_normal.choices[0].message.content.strip()}")
    print(f"  prompt_tokens: {resp_normal.usage.prompt_tokens}")

    print("\nWith safe_prompt=True:")
    print(f"  {resp_safe.choices[0].message.content.strip()}")
    print(f"  prompt_tokens: {resp_safe.usage.prompt_tokens}  ← higher due to injected safety instruction")

except EnvironmentError as e:
    print(f"[safe_prompt] Skipped — set MISTRAL_API_KEY in .env: {e}")

---
## 13. Structured / JSON Output

Free-form text responses are hard to parse reliably. Mistral supports two approaches to get **structured data** back:

### Option A — `response_format={"type": "json_object"}`
Tells the model to always return valid JSON. You still control the shape via the prompt.

### Option B — JSON Schema / Pydantic (SDK v2)
Pass a Pydantic model or a JSON Schema dict to `response_format`. The SDK validates the response against the schema and returns a typed object.

**When to use structured output:**
- Extracting entities, classifying text, filling forms
- Any time downstream code needs to `.get()` or index into the result
- Replacing fragile regex parsing of LLM responses

In [ ]:
import json

try:
    import config
    from llm_client import get_client

    client = get_client()

    # --- Option A: json_object mode ---
    resp = client.chat.complete(
        model=config.MISTRAL_MODEL,
        messages=[{
            "role": "user",
            "content": (
                "Extract the following fields from this sentence and return JSON:\n"
                "name, city, job\n\n"
                "Sentence: 'Marie Curie lived in Paris and worked as a physicist.'"
            ),
        }],
        response_format={"type": "json_object"},
        max_tokens=100,
    )

    raw = resp.choices[0].message.content
    parsed = json.loads(raw)

    print("Option A — json_object mode:")
    print(f"  Raw response:  {raw}")
    print(f"  Parsed Python: {parsed}")
    print(f"  name={parsed.get('name')!r}, city={parsed.get('city')!r}, job={parsed.get('job')!r}")

except EnvironmentError as e:
    print(f"[structured output] Skipped — set MISTRAL_API_KEY in .env: {e}")

In [ ]:
try:
    from pydantic import BaseModel
    from mistralai import Mistral
    import config
    from llm_client import get_client

    client = get_client()

    # --- Option B: Pydantic schema enforcement (SDK v2) ---
    class Review(BaseModel):
        sentiment: str        # "positive" | "neutral" | "negative"
        score: int            # 1–5
        summary: str          # one sentence

    resp = client.chat.complete(
        model=config.MISTRAL_MODEL,
        messages=[{
            "role": "user",
            "content": (
                "Analyse this product review and fill in the schema:\n\n"
                "'The battery life is outstanding and the screen is crisp, "
                "but the price feels a bit steep for what you get.'"
            ),
        }],
        response_format=Review,
        max_tokens=150,
    )

    review: Review = Review.model_validate_json(resp.choices[0].message.content)
    print("Option B — Pydantic schema:")
    print(f"  sentiment : {review.sentiment}")
    print(f"  score     : {review.score}/5")
    print(f"  summary   : {review.summary}")

except ImportError:
    print("Pydantic not installed — run: pip install pydantic")
except EnvironmentError as e:
    print(f"[pydantic output] Skipped — set MISTRAL_API_KEY in .env: {e}")

---
## 14. Function / Tool Calling

Tool calling lets the model decide **when to call a Python function** and **what arguments to pass**. The flow is:

```
1. You define tools (function name + JSON schema of parameters)
2. You send a user message + tools list to the API
3. The model responds with finish_reason="tool_calls" and a list of calls
4. You execute the function(s) and send results back as "tool" role messages
5. The model produces a final natural-language answer using the results
```

This enables the model to access real-time data, perform calculations, or trigger actions that it can't do by itself.

In [ ]:
import json

# --- Step 1: Define the tool (Python function + schema) ---

def get_weather(city: str, unit: str = "celsius") -> dict:
    """Simulated weather API — replace with a real one in production."""
    fake_data = {
        "paris":   {"temp": 18, "condition": "partly cloudy"},
        "london":  {"temp": 14, "condition": "rainy"},
        "new york": {"temp": 22, "condition": "sunny"},
    }
    data = fake_data.get(city.lower(), {"temp": 20, "condition": "unknown"})
    if unit == "fahrenheit":
        data["temp"] = round(data["temp"] * 9 / 5 + 32)
    return {"city": city, "unit": unit, **data}

# JSON schema describing the function to the model
weather_tool = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the current weather for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"], "default": "celsius"},
            },
            "required": ["city"],
        },
    },
}

try:
    import config
    from llm_client import get_client

    client = get_client()
    messages = [{"role": "user", "content": "What's the weather like in Paris right now?"}]

    # --- Step 2: First call — model decides to use the tool ---
    resp1 = client.chat.complete(
        model=config.MISTRAL_MODEL,
        messages=messages,
        tools=[weather_tool],
        tool_choice="auto",
    )

    choice = resp1.choices[0]
    print(f"finish_reason: {choice.finish_reason}")

    if choice.finish_reason == "tool_calls":
        # --- Step 3: Execute the function(s) ---
        tool_calls = choice.message.tool_calls
        messages.append(choice.message)  # append assistant's tool_call message

        for tc in tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)
            print(f"\nModel called: {fn_name}({fn_args})")

            result = get_weather(**fn_args)
            print(f"Function returned: {result}")

            # --- Step 4: Send results back ---
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result),
            })

        # --- Step 5: Final answer ---
        resp2 = client.chat.complete(
            model=config.MISTRAL_MODEL,
            messages=messages,
        )
        print(f"\nFinal answer:\n  {resp2.choices[0].message.content}")

except EnvironmentError as e:
    print(f"[tool calling] Skipped — set MISTRAL_API_KEY in .env: {e}")

---
## 15. Async Calls

For high-throughput use cases (many requests in parallel, integration with async frameworks like FastAPI), use `client.chat.complete_async()`.

**In Jupyter**, the event loop is already running, so you can `await` directly at the top level. In a regular Python script, wrap in `asyncio.run(main())`.

**Why async matters:**
- `asyncio.gather(...)` fans out N requests concurrently — wall time ≈ single request latency instead of N × latency
- Essential for batch processing, real-time applications, and serving multiple users

In [ ]:
import asyncio
import time

async def async_demo():
    import config
    from llm_client import get_client

    client = get_client()

    questions = [
        "What is the capital of France? One word.",
        "What is the capital of Germany? One word.",
        "What is the capital of Japan? One word.",
    ]

    async def ask(question: str) -> str:
        resp = await client.chat.complete_async(
            model=config.MISTRAL_MODEL,
            messages=[{"role": "user", "content": question}],
            max_tokens=10,
        )
        return resp.choices[0].message.content.strip()

    # Sequential for comparison
    t0 = time.perf_counter()
    sequential = []
    for q in questions:
        sequential.append(await ask(q))
    seq_time = time.perf_counter() - t0

    # Concurrent — all 3 requests in-flight at once
    t0 = time.perf_counter()
    concurrent = await asyncio.gather(*[ask(q) for q in questions])
    con_time = time.perf_counter() - t0

    print("Sequential results:")
    for q, a in zip(questions, sequential):
        print(f"  {q!r} → {a}")
    print(f"  Time: {seq_time:.2f}s\n")

    print("Concurrent results (asyncio.gather):")
    for q, a in zip(questions, concurrent):
        print(f"  {q!r} → {a}")
    print(f"  Time: {con_time:.2f}s")
    print(f"\nSpeedup: {seq_time / con_time:.1f}x")

try:
    import config  # trigger EnvironmentError early if no key
    await async_demo()
except EnvironmentError as e:
    print(f"[async] Skipped — set MISTRAL_API_KEY in .env: {e}")

---
## 16. Observability — Token & Cost Tracking

Every `chat()` call logs token counts at INFO level (via the `trace_id`). This section shows how to capture them programmatically for cost estimation and latency monitoring.

**Token cost formula:**
```
cost = (prompt_tokens × price_per_prompt_token) + (completion_tokens × price_per_completion_token)
```

**Latency** = wall-clock time from request sent to full response received (already logged by `llm_client.py`).

For production-grade observability, drop-in integrations are available:

| Tool | One-liner setup | What you get |
|---|---|---|
| **Langfuse** | `from langfuse.mistralai import MistralAI` | Per-request traces, cost dashboard, prompt versioning |
| **MLflow** | `mlflow.mistral.autolog()` | Experiment tracking, model registry, auto-logged traces |
| **Langtrace** | `Langtrace.init(api_key=...)` | OpenTelemetry-based tracing, open source |

In [ ]:
import time

# Approximate prices (USD per 1M tokens) for mistral-large-latest.
# Check https://mistral.ai/technology/#pricing for current rates.
PRICE_PER_M = {"prompt": 2.00, "completion": 6.00}

def cost_usd(prompt_tokens: int, completion_tokens: int) -> float:
    return (
        prompt_tokens    * PRICE_PER_M["prompt"]     / 1_000_000 +
        completion_tokens * PRICE_PER_M["completion"] / 1_000_000
    )

try:
    import config
    from llm_client import get_client

    client = get_client()

    prompts = [
        "Summarise quantum computing in one sentence.",
        "What is the boiling point of water in Celsius?",
        "Name three programming languages invented before 1980.",
    ]

    print(f"{'Prompt':<50} {'p_tok':>6} {'c_tok':>6} {'total':>6} {'lat(s)':>7} {'cost($)':>9}")
    print("-" * 95)

    total_prompt = total_completion = total_cost = 0.0

    for prompt in prompts:
        t0 = time.perf_counter()
        resp = client.chat.complete(
            model=config.MISTRAL_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=60,
        )
        lat = time.perf_counter() - t0
        u = resp.usage
        c = cost_usd(u.prompt_tokens, u.completion_tokens)

        total_prompt     += u.prompt_tokens
        total_completion += u.completion_tokens
        total_cost       += c

        print(f"{prompt[:48]:<50} {u.prompt_tokens:>6} {u.completion_tokens:>6} "
              f"{u.total_tokens:>6} {lat:>7.2f} {c:>9.6f}")

    print("-" * 95)
    print(f"{'TOTAL':<50} {int(total_prompt):>6} {int(total_completion):>6} "
          f"{int(total_prompt+total_completion):>6} {'':>7} {total_cost:>9.6f}")

except EnvironmentError as e:
    print(f"[observability] Skipped — set MISTRAL_API_KEY in .env: {e}")

---
## 17. RAG — Retrieval-Augmented Generation

**The problem:** LLMs have a knowledge cutoff and no access to your private data. Asking about internal documents, recent events, or proprietary knowledge will produce hallucinations.

**The solution:** RAG grounds the model's answer in retrieved documents:

```
Documents → embed → vector store
                         ↓
Query → embed → find top-k similar chunks → inject as context → generate grounded answer
```

### This demo uses
- **`mistral-embed`** — Mistral's embedding model (1024-dimensional vectors)
- **In-memory vector store** — a plain Python list; replace with Chroma, Qdrant, pgvector, etc. for production
- **Cosine similarity** — no extra dependencies, pure Python
- **`mistral-large-latest`** — for the final answer generation

### Three phases
1. **Indexing** — embed your knowledge base (done once, offline)
2. **Retrieval** — embed the query, find the most similar chunks
3. **Generation** — pass the retrieved context + question to the LLM

In [ ]:
import math

# ---------------------------------------------------------------------------
# Knowledge base — small corpus about Mistral AI (stand-in for your documents)
# In production: load PDFs, database rows, website pages, etc.
# ---------------------------------------------------------------------------
DOCUMENTS = [
    {
        "id": "doc1",
        "text": (
            "Mistral AI was founded in April 2023 by Arthur Mensch, Guillaume Lample, "
            "and Timothée Lacroix, all former researchers at DeepMind and Meta. "
            "The company is headquartered in Paris, France."
        ),
    },
    {
        "id": "doc2",
        "text": (
            "Mistral 7B was released in September 2023 as an open-weight model under "
            "the Apache 2.0 licence. It outperformed Llama 2 13B on most benchmarks "
            "despite having fewer parameters, thanks to grouped-query attention and "
            "sliding window attention."
        ),
    },
    {
        "id": "doc3",
        "text": (
            "Mixtral 8x7B is a sparse mixture-of-experts model released by Mistral AI "
            "in December 2023. It uses 8 expert networks but only activates 2 per token, "
            "giving the capacity of a 47B model at the inference cost of a 13B model."
        ),
    },
    {
        "id": "doc4",
        "text": (
            "Mistral Large is Mistral AI's flagship commercial model. It supports a "
            "32k token context window, excels at complex reasoning, multi-step tasks, "
            "and code generation, and is available via the La Plateforme API."
        ),
    },
    {
        "id": "doc5",
        "text": (
            "La Plateforme is Mistral AI's API platform. It offers both open-weight "
            "models (Mistral 7B, Mixtral 8x7B) and proprietary models (Mistral Small, "
            "Mistral Medium, Mistral Large). Pricing is per token consumed."
        ),
    },
    {
        "id": "doc6",
        "text": (
            "Mistral AI raised a €385 million Series B round in June 2024, valuing "
            "the company at €6 billion. Investors include Andreessen Horowitz, "
            "General Catalyst, and Nvidia."
        ),
    },
]

print(f"Knowledge base: {len(DOCUMENTS)} documents")
for doc in DOCUMENTS:
    print(f"  [{doc['id']}] {doc['text'][:80]}…")

In [ ]:
# ---------------------------------------------------------------------------
# Utility functions — no external dependencies
# ---------------------------------------------------------------------------

def cosine_similarity(a: list[float], b: list[float]) -> float:
    """Cosine similarity between two embedding vectors."""
    dot   = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0


def embed(client, texts: list[str]) -> list[list[float]]:
    """Embed a list of texts using mistral-embed and return a list of vectors."""
    resp = client.embeddings.create(model="mistral-embed", inputs=texts)
    return [item.embedding for item in resp.data]


def retrieve(query_vec: list[float], index: list[dict], top_k: int = 3) -> list[dict]:
    """Return the top-k most similar documents to the query vector."""
    scored = [
        {**doc, "score": cosine_similarity(query_vec, doc["embedding"])}
        for doc in index
    ]
    return sorted(scored, key=lambda d: d["score"], reverse=True)[:top_k]


print("Utility functions defined: cosine_similarity, embed, retrieve")

### Phase 1 — Indexing

Embed all documents once and store the vectors alongside the text. In production this is done offline and the index is persisted to a vector database.

In [ ]:
try:
    from llm_client import get_client

    client = get_client()

    # Embed all documents in a single API call (batching is cheaper and faster)
    texts = [doc["text"] for doc in DOCUMENTS]
    vectors = embed(client, texts)

    # Build the in-memory index: each entry = document metadata + its embedding
    INDEX = [
        {**doc, "embedding": vec}
        for doc, vec in zip(DOCUMENTS, vectors)
    ]

    print(f"Indexed {len(INDEX)} documents")
    print(f"Embedding dimensions: {len(INDEX[0]['embedding'])}")
    print(f"\nSample vector (first 8 dimensions of doc1):")
    print(f"  {INDEX[0]['embedding'][:8]}")

except EnvironmentError as e:
    INDEX = []
    print(f"[RAG indexing] Skipped — set MISTRAL_API_KEY in .env: {e}")

### Phase 2 — Retrieval

Embed the user's query with the same model, then rank all documents by cosine similarity and return the top-k most relevant chunks.

In [ ]:
if INDEX:
    query = "Who founded Mistral AI and where is the company based?"
    print(f"Query: {query!r}\n")

    # Embed the query with the same model used for indexing (critical!)
    query_vec = embed(client, [query])[0]

    # Rank all documents and return top-3
    top_docs = retrieve(query_vec, INDEX, top_k=3)

    print("Top-3 retrieved documents (by cosine similarity):")
    for i, doc in enumerate(top_docs, 1):
        print(f"\n  [{i}] id={doc['id']}  score={doc['score']:.4f}")
        print(f"       {doc['text']}")
else:
    print("[retrieval] Skipped — index is empty (no API key)")

### Phase 3 — Generation

Inject the retrieved chunks as context into the prompt. The model answers based on the context, not (just) its training data — reducing hallucination and enabling answers about private or recent information.

In [ ]:
if INDEX:
    # Build the context block from retrieved documents
    context = "\n\n".join(
        f"[Source {i+1}]\n{doc['text']}"
        for i, doc in enumerate(top_docs)
    )

    rag_prompt = (
        "Answer the question using only the sources below. "
        "If the answer is not in the sources, say 'I don't know'.\n\n"
        f"Sources:\n{context}\n\n"
        f"Question: {query}"
    )

    print("RAG prompt sent to the model:")
    print("-" * 60)
    print(rag_prompt)
    print("-" * 60)

    resp = client.chat.complete(
        model=config.MISTRAL_MODEL,
        messages=[{"role": "user", "content": rag_prompt}],
        max_tokens=200,
    )

    print(f"\nAnswer:\n{resp.choices[0].message.content}")
    print(f"\n(used {resp.usage.total_tokens} tokens total)")
else:
    print("[RAG generation] Skipped — index is empty (no API key)")

### End-to-end RAG function

The three phases above wrapped into a single reusable function.

In [ ]:
def rag_answer(question: str, index: list[dict], top_k: int = 3) -> str:
    """Full RAG pipeline: embed query → retrieve → generate."""
    query_vec = embed(client, [question])[0]
    docs = retrieve(query_vec, index, top_k=top_k)

    context = "\n\n".join(
        f"[Source {i+1}]\n{d['text']}" for i, d in enumerate(docs)
    )
    prompt = (
        "Answer the question using only the sources below. "
        "If the answer is not in the sources, say 'I don't know'.\n\n"
        f"Sources:\n{context}\n\n"
        f"Question: {question}"
    )
    resp = client.chat.complete(
        model=config.MISTRAL_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200,
    )
    return resp.choices[0].message.content.strip()


if INDEX:
    test_questions = [
        "What is Mixtral 8x7B and how does it work?",
        "How much money has Mistral AI raised?",
        "What is the weather like in Paris?",   # out-of-scope — should say "I don't know"
    ]

    for q in test_questions:
        print(f"Q: {q}")
        print(f"A: {rag_answer(q, INDEX)}\n")
else:
    print("[RAG function] Skipped — index is empty (no API key)")

### RAG Playground

Edit the question below and try your own queries against the knowledge base.

In [ ]:
# ✏️ Edit this question and re-run the cell
MY_QUESTION = "What models does Mistral offer and where can I access them?"
TOP_K = 3  # number of documents to retrieve

if INDEX:
    answer = rag_answer(MY_QUESTION, INDEX, top_k=TOP_K)
    print(f"Q: {MY_QUESTION}\n")
    print(f"A: {answer}")
else:
    print("[RAG playground] Set MISTRAL_API_KEY in .env to use this cell")